# Centipede: physics options on a Kaggle T4

This notebook runs `benchmarks/physics_options.py` from the public repository.
It changes the model's physics options in memory only (the model file is never
changed) and, for each variant, measures the time of one 20 ms action at 1,024
worlds, launched from Python and replayed as a CUDA graph, and checks that the
physics stays sound: the deepest contact penetration, the largest joint-limit
excess, settling with motors off compared with the model as it is, and the
average height and speed of the body under random leg commands. The variants
are Newton and CG, two solver tolerances, a shorter line search, pyramidal
friction cones, and timesteps of 0.125, 0.149, and 0.2 ms.

Before running, in the notebook's side panel:

- **Accelerator:** GPU T4 (one is enough).
- **Internet:** on.

Then choose **Run all**. It takes roughly 30 to 40 minutes. The results are
saved in `/kaggle/working/physics_options` (`report.txt` and
`physics_options.json`) and zipped into `/kaggle/working/physics_options.zip`.

In [ ]:
# The machine: GPUs, driver, CPU cores, and Python version.
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!nproc
!python --version

In [ ]:
# Fetch the project and the two sibling packages it installs. Change BRANCH
# to profile another branch of Centipede.
OWNER = "https://github.com/DrippingSoup22"
BRANCH = "main"

!rm -rf /kaggle/working/centipede /kaggle/working/RL_lib /kaggle/working/MujocoReplay
!git clone --depth 1 --branch {BRANCH} {OWNER}/centipede-project.git /kaggle/working/centipede
!git clone --depth 1 {OWNER}/RL_lib.git /kaggle/working/RL_lib
!git clone --depth 1 {OWNER}/MujocoReplay.git /kaggle/working/MujocoReplay
%cd /kaggle/working/centipede
!git log -1 --format="Centipede commit %h: %s"
!git -C /kaggle/working/RL_lib log -1 --format="RL_lib commit %h: %s"
!git -C /kaggle/working/MujocoReplay log -1 --format="MujocoReplay commit %h: %s"

In [ ]:
# Install RL_lib (the learning algorithms) and MujocoReplay (which writes the
# replay recordings), then the package with the GPU backend. Kaggle
# already provides PyTorch with CUDA, so it is not downloaded again.
!pip install --quiet -e /kaggle/working/RL_lib -e /kaggle/working/MujocoReplay
!pip install --quiet -e ".[gpu]"

In [ ]:
# The versions in use, and the GPU as PyTorch and Warp see it.
import mujoco
import mujoco_warp
import torch
import warp

print("MuJoCo", mujoco.__version__, "| MuJoCo Warp", mujoco_warp.__version__)
print("Warp", warp.__version__, "| PyTorch", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(),
    "| compute capability",
    torch.cuda.get_device_capability(),
    "|",
    torch.cuda.device_count(),
    "visible",
)
assert torch.cuda.get_device_capability() >= (7, 0), "Choose the T4 accelerator"

In [ ]:
# Every variant, compared with the model as it is.
!python benchmarks/physics_options.py --output /kaggle/working/physics_options

In [ ]:
# Keep the results in one file for download.
!cd /kaggle/working && zip -qr physics_options.zip physics_options && ls -lh physics_options.zip